In [2]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error, r2_score
import plotly.express as px
import plotly.graph_objects as go
import warnings
warnings.filterwarnings('ignore')

# =====================================================================
# PART 1: COMPUTE FORECAST MODEL KPIs
# =====================================================================
# 1. Read prediction data (paths updated to features folder)
df_pred = pd.read_csv('../features/stage_4_input_predictions.csv')
df_pred.columns = df_pred.columns.str.strip()

# Standardize actual column name
column_mapping = {'quantity': 'actual_quantity', 'Quantity': 'actual_quantity', 'Actual': 'actual_quantity', 'actual': 'actual_quantity'}
df_pred = df_pred.rename(columns=column_mapping)
df_pred['order_date'] = pd.to_datetime(df_pred['order_date'])

# Automatically detect prediction columns (starting with 'pred_')
pred_cols = [col for col in df_pred.columns if str(col).startswith('pred_')]
models = {col.replace('pred_', '').replace('_', ' ').title(): col for col in pred_cols}

# 2. Compute RMSE and R2 Score
metrics_data = []
for model_name, col_name in models.items():
    valid_data = df_pred.dropna(subset=['actual_quantity', col_name])
    if not valid_data.empty:
        rmse = np.sqrt(mean_squared_error(valid_data['actual_quantity'], valid_data[col_name]))
        r2 = r2_score(valid_data['actual_quantity'], valid_data[col_name])
        metrics_data.append({'Model': model_name, 'RMSE': round(rmse, 2), 'R2 Score': round(r2, 4)})

# Create DataFrame and rank by lowest RMSE
df_metrics = pd.DataFrame(metrics_data).sort_values(by='RMSE')

# 3. Export CSV (save into outputs folder)
output_file = '../src/outputs/reports/model_evaluation_metrics.csv'
df_metrics.to_csv(output_file, index=False)

print(f"✅ Saved model evaluation results to: {output_file}")
display(df_metrics) # Show table in Jupyter

✅ Saved model evaluation results to: ../src/outputs/reports/model_evaluation_metrics.csv


,Model,RMSE,R2 Score
0,Multiple Linear,58.34,0.2315
1,Ridge Regression,58.34,0.2315
4,Xgboost,58.71,0.2215
2,Random Forest,58.74,0.2209
3,Lightgbm,59.40,0.2033
5,Lstm,63.53,0.0885
6,Naive Baseline,86.91,-0.7058


In [ ]:
# =====================================================================
# CHART PLOTTING (UPDATED: SHOW ALL MODELS)
# =====================================================================
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from sklearn.metrics import mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

# Rebuild required inputs when this cell is run independently
if 'df_pred' not in globals():
    df_pred = pd.read_csv('../features/stage_4_input_predictions.csv')
    df_pred.columns = df_pred.columns.str.strip()
    df_pred['order_date'] = pd.to_datetime(df_pred['order_date'])

column_mapping = {'quantity': 'actual_quantity', 'Quantity': 'actual_quantity', 'Actual': 'actual_quantity', 'actual': 'actual_quantity'}
if 'actual_quantity' not in df_pred.columns:
    df_pred = df_pred.rename(columns={k: v for k, v in column_mapping.items() if k in df_pred.columns})

pred_cols = [col for col in df_pred.columns if str(col).startswith('pred_')]
if not pred_cols:
    raise NameError("No prediction columns found in df_pred (expected columns starting with 'pred_').")
models = {col.replace('pred_', '').replace('_', ' ').title(): col for col in pred_cols}

if 'df_metrics' not in globals():
    try:
        df_metrics = pd.read_csv('../src/outputs/reports/model_evaluation_metrics.csv')
    except Exception:
        metrics_data = []
        for model_name, col_name in models.items():
            valid_data = df_pred.dropna(subset=['actual_quantity', col_name])
            if not valid_data.empty:
                rmse = np.sqrt(mean_squared_error(valid_data['actual_quantity'], valid_data[col_name]))
                r2 = r2_score(valid_data['actual_quantity'], valid_data[col_name])
                metrics_data.append({'Model': model_name, 'RMSE': round(rmse, 2), 'R2 Score': round(r2, 4)})
        df_metrics = pd.DataFrame(metrics_data).sort_values(by='RMSE')

if 'stock_code' not in df_pred.columns:
    raise NameError("df_pred must contain 'stock_code' column to select top SKU.")

if 'df_filtered' not in globals():
    top_sku = df_pred['stock_code'].value_counts().index[0]
    df_filtered = df_pred[df_pred['stock_code'] == top_sku].sort_values(by='order_date')
else:
    top_sku = df_filtered['stock_code'].iloc[0]

print("\n--- 📈 3. ACTUAL VS FORECAST PLOT (example uses top-selling SKU) ---")

best_model_name = df_metrics.iloc[0]['Model']
best_model_col = models[best_model_name]

fig_line = go.Figure()

# 1. Plot the Actual line - solid, thick, green
fig_line.add_trace(go.Scatter(
    x=df_filtered['order_date'], 
    y=df_filtered['actual_quantity'], 
    name='Actual', 
    line=dict(color='#2ca02c', width=3),
))

# Color palette for automatic model assignment
colors = ['#1f77b4', '#ff7f0e', '#9467bd', '#8c564b', '#e377c2', '#17becf', '#7f7f7f']

# 2. Loop through all models and plot them
for i, (model_name, col_name) in enumerate(models.items()):
    # Highlight XGBoost in red with a stronger dotted line
    if "Xgboost" in model_name or "Xgb" in model_name:
        line_style = dict(color='red', width=2.5, dash='dot')
        trace_name = model_name
    else:
        line_style = dict(color=colors[i % len(colors)], width=1.5, dash='dash')
        trace_name = model_name

    fig_line.add_trace(go.Scatter(
        x=df_filtered['order_date'], 
        y=df_filtered[col_name], 
        name=trace_name, 
        line=line_style
    ))

# 3. Customize the layout for a clean presentation
fig_line.update_layout(
    title=f"Forecast vs Actual for SKU: {top_sku} (All Models)",
    xaxis_title="Time (Order Date)",
    yaxis_title="Quantity",
    hovermode="x unified",
    legend=dict(
        orientation="h",
        yanchor="top",
        y=-0.2,
        xanchor="center",
        x=0.5
    ),
    template="plotly_white",
)

# Export the figure to file (PNG only)
import os
output_dir = '../src/outputs/images'
os.makedirs(output_dir, exist_ok=True)
png_path = os.path.join(output_dir, f'forecast_vs_actual_{top_sku}.png')
try:
    fig_line.write_image(png_path, scale=2)
    print(f"✅ Saved PNG: {png_path}")
except Exception as exc:
    print(f"⚠️ Could not save PNG (kaleido missing or export failed): {exc}")

fig_line.show()



--- 📈 3. ACTUAL VS FORECAST PLOT (example uses top-selling SKU) ---
✅ Saved PNG: ../src/ouputs/images\forecast_vs_actual_16237.png
